# EDA na Prática

Nas aulas 9 e 10 vimos os quatro princípios da EDA (estrutural, distribucional, relacional e detecção de outliers/anomalias), como tratar dados ausentes e outliers, e como fazer análise descritiva, bivariada e multivariada. Hoje é a hora de **aplicarmos na prática**. Durante a primeira hora da aula, serão apresentadas as técnicas no dataset *Heart Disease* (UCI) e depois **cada aluno** irá aplicar as técnicas no dataset que foi escolhido semana passada.

*Relembrando*: a EDA não é uma linha reta

A EDA costuma ser apresentada como uma sequência de etapas (estrutura → tratamento → descrição → relação), mas na prática ela é **iterativa**: às vezes tratamos um outlier, olhamos a estatística descritiva de novo, e percebemos que precisa voltar atrás. Hoje vamos deixar isso explícito: olhamos a estatística descritiva **antes** de tratar dados ausentes/outliers (pra saber *como* tratar) e **depois** (pra confirmar o efeito do tratamento).

## Como funcionará a atividade

- **Duração:** ~1h40min
- **Formato:** individual
- **Passo 1:** acompanhem a demonstração no dataset Heart Disease.
- **Passo 2:** importem o dataset que vocês escolheram e enviaram anteriormente.
- **Passo 3:** repliquem o mesmo ciclo no dataset escolhido, respondendo às perguntas guiadas.
- **Uso de IA:** permitido, conforme o Plano de Ensino — documentem na seção final.

---

**Este notebook contém a réplica da atividade aplicada ao meu dataset escolhido: [EV Battery Health Prediction Dataset (Kaggle)](https://www.kaggle.com/datasets/srisyra02/ev-battery-health-prediction-dataset-20k)**, que registra 70 atributos de telemetria, cadastro e diagnóstico de 20.000 veículos elétricos, com o objetivo de prever falha de bateria (`battery_failure`).


## Setup de Bibliotecas e Dados


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "lightgray"     # convenção: cinza-claro pra tudo que é "pano de fundo" genérico
COR_CONTEXTO = "#e9d4d0"    # rosa-claro pra pano de fundo / classe de referência
COR_DESTAQUE = "#2d223a"    # roxo pra classe de interesse / "o personagem da história"
COR_ALERTA = "tab:red"       # exceção deliberada: outliers/pontos de atenção


## Carregando o Dataset


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(   # note: dataset_load, não load_dataset (que está deprecated)
    KaggleDatasetAdapter.PANDAS,
    "srisyra02/ev-battery-health-prediction-dataset-20k",
    "ev battery_failure prediction Dataset.csv",  # <- nome exato do arquivo dentro do dataset
)
print("First 5 records:")
df.head()


## 1. Princípio Estrutural

Antes de qualquer análise, precisamos entender a "anatomia" dos dados: quantas linhas e colunas temos, que tipo de dado cada coluna guarda, se há valores faltantes e se há duplicadas.

**Antes de rodar:** este dataset tem 70 colunas, misturando dados de cadastro do veículo (marca, modelo, ano), métricas de telemetria da bateria (tensão, temperatura, ciclos de carga) e scores de diagnóstico (saúde térmica, qualidade de carregamento), além do alvo `battery_failure` (0 = sem falha, 1 = falha). Com 20 mil veículos catalogados, quantas linhas vocês esperam que restem depois de tratar ausentes e duplicadas — próximo de 20 mil, ou uma queda considerável?


### Dimensão e Tipos de Dados


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Duplicadas


In [ ]:
print("Linhas totalmente duplicadas:", df.duplicated().sum())

# Como vehicle_id deveria ser uma chave única por veículo, vale checar separadamente
print("vehicle_id duplicado:", df["vehicle_id"].duplicated().sum())


### Criando um rótulo legível para o atributo alvo


O dataset já registra o alvo `battery_failure` como binário (0 = sem falha, 1 = falha) — diferente do Heart Disease, aqui não precisamos simplificar nada. Vamos apenas criar uma versão legível, `status_bateria`, para usar nos títulos e legendas dos gráficos ao longo da aula (rosa = sem falha, classe de referência; roxo = com falha, classe de interesse). Mantemos a coluna numérica original, já que ela é útil para correlações e testes estatísticos mais à frente.


In [ ]:
df["status_bateria"] = np.where(df["battery_failure"] == 1, "Falha", "Normal")
df["status_bateria"].value_counts()


## 2. Dados Faltantes


In [ ]:
df.isna().sum()

**Discussao:** Alguma feature tem dados ausentes nos seus dados?

*Resposta*: Sim, diversas features apresentam dados ausentes — de um total de 70 colunas, 67 delas têm algum grau de ausência (apenas `vehicle_id` e a variável-alvo `battery_failure`, entre outras poucas, estão completas). As mais afetadas, em termos percentuais, são `fast_charge_ratio` (4.94%), `average_speed` (4.92%) e `manufacturing_year` (4.91%), seguidas de perto por várias outras métricas de bateria e sensores (`sensor_fault_count`, `state_of_charge`, `state_of_health`, `capacity_loss_percent`), todas na faixa de 4.6% a 4.9%. A variável-alvo `battery_failure` não apresenta ausências, o que é positivo para a modelagem. Vale notar que os percentuais estão bastante próximos entre si e relativamente baixos (nenhuma coluna passa de 5%), sem nenhuma se destacar com uma taxa muito acima das demais — um padrão mais consistente com ausência MCAR (completamente aleatória) do que com um mecanismo sistemático ligado a outra variável, mas essa hipótese será melhor investigada na etapa relacional da EDA

In [ ]:
(df.isna().sum() > 0).sum()

#### Identificando o mecanismo de dados ausentes


**Conhecimento de Domínio**

No dataset atual, 67 das 70 features têm dados ausentes, todos numa faixa relativamente estreita (~4.6% a 4.94%).

Como os dados foram coletados: dados de veículos elétricos costumam vir de fontes heterogêneas — telemetria do veículo (sensores embarcados), registros do fabricante (ano, modelo, química da bateria) e testes de laboratório/diagnóstico (scores de saúde, capacidade, eficiência). Cada fonte tem sua própria taxa de falha de captura.

Qual a natureza da fonte de dados: como o dataset mistura variáveis estáticas (ex: `manufacturing_year`, `vehicle_brand`) com variáveis calculadas/derivadas de sensores (ex: `sensor_fault_count`, `thermal_health_score`), é plausível que a ausência tenha mais de uma causa: falhas de sensor (afetando métricas de telemetria) e lacunas de cadastro (afetando dados do veículo).

**As ausencias ocorrem juntas?**

Contar ausentes por coluna não conta a história toda: se duas colunas ficam ausentes **nas mesmas linhas**, isso é um indício de padrão sistemático (ex: um exame que não foi feito, afetando várias medidas de uma vez).



In [ ]:
cols_com_ausentes = df.columns[df.isna().any()].tolist()
print(f"Total de colunas com ausentes: {len(cols_com_ausentes)}")

# Matriz binária: True/1 onde o dado está ausente
matriz_ausencia = df[cols_com_ausentes].isna().astype(int)

# Correlação entre os padrões de ausência (não entre os valores!)
corr_ausencia = matriz_ausencia.corr()

plt.figure(figsize=(18, 15))
sns.heatmap(corr_ausencia, cmap="magma", center=0, square=True,
            cbar_kws={"label": "Correlação de ausência"})
plt.title("Correlação entre padrões de ausência (67 colunas)")
plt.tight_layout()
plt.show()


Como são 67 colunas, o heatmap completo fica poluído — vale complementar buscando os pares mais fortes:

In [ ]:
# Pega os pares de colunas com maior correlação de ausência (fora da diagonal)
corr_pares = corr_ausencia.where(np.triu(np.ones(corr_ausencia.shape), k=1).astype(bool))
corr_pares = corr_pares.stack().sort_values(ascending=False)
print("Top 10 pares com ausência mais correlacionada:")
print(corr_pares.head(10))


**Pra discutir:** existem pares (ou grupos) de colunas cuja ausência está fortemente correlacionada — sugerindo que um mesmo evento (ex: falha de um sensor, um registro de cadastro incompleto) derruba várias métricas ao mesmo tempo? Ou os 67 padrões de ausência parecem, em sua maioria, independentes entre si (mais consistente com MCAR)?

Não. Mesmo com 67 colunas e mais de 2.200 pares possíveis de comparação, o par com maior correlação de ausência (pack_voltage × daily_distance) atinge apenas 0.026, e o restante do top 10 fica entre 0.021 e 0.026 — valores próximos de zero, sem nenhum indício de padrão sistemático. Isso é bem diferente do caso do Heart Disease, onde poucas colunas ausentes (ca, thal) tornavam qualquer coeficiente pouco confiável; aqui temos volume suficiente (67 colunas, milhares de linhas) para confiar no resultado, e ele aponta que a ausência ocorre de forma independente entre as colunas. Isso é consistente com a hipótese de MCAR (mecanismo aleatório e sem viés): não parece existir um "evento comum" (ex: um sensor que falha e derruba várias métricas juntas) causando os ausentes — cada coluna parece perder dados de forma isolada, provavelmente por ruído/erro de captura em sensores e registros individuais, e não por um problema estrutural do processo de coleta.

**Nota para quando vocês replicarem no dataset de vocês:** se o dataset escolhido tiver *muitas* colunas com *muitos* ausentes, aí sim uma correlação (ou um heatmap) entre os indicadores de ausência passa a fazer sentido. O problema neste caso foi o volume pequeno, não a técnica em si.

## 3. Princípio Distribucional

Ainda com os dados originais (sem tratar ausentes ou outliers), vamos calcular as estatísticas descritivas e olhar as distribuições. Esse retrato "antes" é o que vai *orientar as decisões de tratamento* na próxima seção. Por exemplo, se uma variável for muito assimétrica, faz mais sentido imputar com a mediana do que com a média.

**Escolha das variáveis:** para manter o mesmo formato da demonstração (5 quantitativas + algumas qualitativas), selecionamos um subconjunto de colunas ligadas à saúde da bateria, em vez de usar as 70 de uma vez:
- **Quantitativas:** `state_of_health` (saúde da bateria, %), `cycle_count` (nº de ciclos de carga), `internal_resistance` (resistência interna), `capacity_loss_percent` (perda de capacidade, %) e `thermal_runaway_risk` (score de risco térmico).
- **Qualitativas:** `vehicle_type`, `battery_chemistry`, `drive_type`, `fleet_or_private`, `terrain_type` e `battery_manufacturer`.


In [ ]:
VARS_QUANT = ["state_of_health", "cycle_count", "internal_resistance", "capacity_loss_percent", "thermal_runaway_risk"]
VARS_QUALI = ["vehicle_type", "battery_chemistry", "drive_type", "fleet_or_private", "terrain_type", "battery_manufacturer"]

### Tabela descritiva, com assimetria e curtose

desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="magma", subset=["skewness", "kurtosis"])
)


**Pra discutir:** olhando a tabela, quais variáveis já parecem assimétricas (`skewness` acima de 1) ou com cauda pesada (`kurtosis` maior que 2 em módulo)?

Isso é coerente com o que vocês esperariam do domínio (ex: a maioria das baterias em operação normal deveria ter risco térmico baixo, com poucos casos extremos puxando a cauda direita de `thermal_runaway_risk`)?


### Distribuições de dados

Onde fizer sentido, comparamos "Normal" (rosa) vs. "Falha" (roxo). Vamos utilizar esta convenção de cor que vamos manter até a análise bivariada.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.ravel()

for i, col in enumerate(VARS_QUANT):
    sns.histplot(data=df, x=col, kde=True, color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)

for ax in axes[len(VARS_QUANT):]:
    ax.axis("off")

fig.suptitle("Distribuição das Variáveis Quantitativas: EV Battery Health (sem separar por classe)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(VARS_QUANT), 2, figsize=(10, 3.5 * len(VARS_QUANT)), sharey=True)

for i, col in enumerate(VARS_QUANT):
    lim = (df[col].min(), df[col].max())
    for j, (classe, cor) in enumerate([("Normal", COR_CONTEXTO), ("Falha", COR_DESTAQUE)]):
        subset = df[df["status_bateria"] == classe]
        sns.histplot(data=subset, x=col, kde=True, color=cor, ax=axes[i, j])
        axes[i, j].set_xlim(lim)
        axes[i, j].set_title(f"{col} — {classe}")

fig.suptitle("Distribuição das Variáveis Quantitativas: EV Battery Health (por classe, em painéis separados)", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()


**Para discutir:** agora que as classes estão em painéis separados (mesma escala de eixo X em cada linha), fica mais fácil ver: alguma variável parece ter formato bem diferente entre "Normal" e "Falha"? Lembrem que a classe "Falha" é bem menor (cerca de 7% das linhas) — isso deixa os histogramas dela mais "ruidosos"/irregulares, o que é esperado com menos observações.


#### Testes de Aderência

*Será se o risco de thermal runaway segue uma distribuição normal?*

O **Shapiro-Wilk** como o teste de aderência mais usado pra normalidade, junto com o **QQ-Plot** como visualização de apoio. Vamos testar com `thermal_runaway_risk`, a variável mais assimétrica da nossa tabela.


In [ ]:
stat_sw, p_valor_sw = stats.shapiro(df["thermal_runaway_risk"].dropna())
print(f"Shapiro-Wilk — thermal_runaway_risk: estatística={stat_sw:.4f}, p-valor={p_valor_sw:.4g}")
print("Rejeita H0 (normalidade)?", "Sim" if p_valor_sw < 0.05 else "Não", "(alpha = 0.05)")

'''### QQ-Plot — risco de thermal runaway'''

fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df["thermal_runaway_risk"].dropna(), dist="norm", plot=ax)
ax.get_lines()[0].set_markerfacecolor(COR_NEUTRA)
ax.get_lines()[0].set_markeredgecolor(COR_NEUTRA)
ax.get_lines()[1].set_color(COR_ALERTA)
ax.set_title("QQ-Plot: Risco de Thermal Runaway vs. Distribuição Normal")
plt.tight_layout()
plt.show()


**Como ler:** se os pontos (cinza) seguissem a reta vermelha de perto, a variável seguiria uma normal. Pontos que se afastam da reta nas pontas indicam cauda mais pesada ou mais leve que a normal — coerente com o resultado do Shapiro-Wilk acima.

**Nota: cuidado com duas armadilhas de interpretação**

- **Tamanho da amostra e poder do teste:** é comum achar que "poucos dados" tornam mais fácil rejeitar a normalidade e é o contrário. Amostras **pequenas** têm **menos poder estatístico**, ou seja, tendem a *não* rejeitar mesmo quando os dados realmente não são normais. Já amostras **muito grandes** (milhares de linhas) deixam o teste sensível demais, rejeitando por desvios triviais. Aqui, com n≈19.000 (bem diferente do n≈303 do Heart Disease), o Shapiro-Wilk tende a rejeitar a normalidade de **praticamente qualquer variável**, mesmo desvios pequenos e sem relevância prática — então o resultado do teste sozinho é menos informativo do que no exemplo anterior. É por isso que olhamos o skewness e o QQ-Plot **junto** com o p-valor: se as três evidências apontam na mesma direção (assimetria alta, QQ-Plot afastado da reta, Shapiro rejeitando), a conclusão é sólida; se só o Shapiro rejeita mas o QQ-Plot está bem próximo da reta, o resultado é provavelmente um artefato do tamanho da amostra.
- **Como afirmar o resultado:** quando o teste **rejeita** H0 (nosso caso, p < 0.05), a afirmação correta é "temos evidência estatística **contra** a normalidade" que é uma conclusão forte. A frase "não podemos garantir normalidade" cabe no caso oposto: quando o teste **não rejeita** H0, já que a ausência de rejeição nunca prova que os dados são normais, só que não há evidência suficiente contra.


In [ ]:
resultados_shapiro = []
for col in VARS_QUANT:
    stat, p = stats.shapiro(df[col].dropna())
    resultados_shapiro.append({"variavel": col, "estatistica": round(stat, 4),
                                "p_valor": round(p, 6), "rejeita_normalidade": p < 0.05})

pd.DataFrame(resultados_shapiro).set_index("variavel")


**ECDF — outra forma de ver a distribuição**

Diferente do histograma, a ECDF não depende de escolher o número de bins: para cada valor no eixo X, a curva mostra a **proporção acumulada** de observações menores ou iguais àquele valor. É especialmente útil pra comparar duas distribuições lado a lado — onde as curvas se distanciam mais é onde a diferença entre os grupos é mais perceptível.

Vamos ilustrar com `thermal_runaway_risk`, só pra fixar como essa visualização se lê.


In [ ]:
plt.figure(figsize=(8, 5))
sns.ecdfplot(data=df, x="thermal_runaway_risk", hue="status_bateria",
             palette={"Normal": COR_CONTEXTO, "Falha": COR_DESTAQUE})
plt.title("ECDF — Risco de Thermal Runaway por Status da Bateria")
plt.xlabel("Score de Risco de Thermal Runaway")
plt.ylabel("Proporção acumulada")
plt.tight_layout()
plt.show()


**Como ler:** em qualquer ponto do eixo X, a altura da curva diz "que fração dos veículos desse grupo tem risco térmico até aqui". Onde as duas curvas (rosa e roxa) se distanciam mais, ali está a faixa de risco onde os dois grupos mais diferem, sem precisar decidir uma largura de bin como no histograma.


### Métricas de Tendência Central


In [ ]:
media_risco = df["thermal_runaway_risk"].mean()
mediana_risco = df["thermal_runaway_risk"].median()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="thermal_runaway_risk", kde=True, color=COR_NEUTRA)
plt.axvline(media_risco, color=COR_ALERTA, linestyle="--", linewidth=2, label=f"Média = {media_risco:.1f}")
plt.axvline(mediana_risco, color="black", linestyle="-", linewidth=2, label=f"Mediana = {mediana_risco:.1f}")
plt.title("Risco de Thermal Runaway: Distribuição com Média e Mediana")
plt.legend()
plt.tight_layout()
plt.show()


### Dispersão


In [ ]:
dispersao = df[VARS_QUANT].agg(["var", "std"]).T
dispersao["IQR"] = df[VARS_QUANT].quantile(0.75) - df[VARS_QUANT].quantile(0.25)
dispersao.round(2)

'''### Boxplot e violin plot por classe'''

palette_classe = {"Normal": COR_CONTEXTO, "Falha": COR_DESTAQUE}

fig, axes = plt.subplots(2, len(VARS_QUANT), figsize=(4 * len(VARS_QUANT), 9), sharex="col")

for i, col in enumerate(VARS_QUANT):
    sns.boxplot(data=df, x="status_bateria", y=col, hue="status_bateria",
                palette=palette_classe, legend=False, ax=axes[0, i])
    axes[0, i].set_title(col)
    axes[0, i].set_xlabel("")

    sns.violinplot(data=df, x="status_bateria", y=col, hue="status_bateria",
                    palette=palette_classe, legend=False, ax=axes[1, i])
    axes[1, i].set_xlabel("")

axes[0, 0].set_ylabel("Boxplot")
axes[1, 0].set_ylabel("Violin plot")
fig.suptitle("Dispersão por Classe — Boxplot vs. Violin Plot", fontsize=14)
plt.tight_layout()
plt.show()


**Pra discutir:** alguma variável mostra uma caixa (IQR) nitidamente mais alta ou mais baixa entre "Normal" e "Falha"? Isso já é um indício — antes mesmo da análise bivariada formal — de que essa variável pode ajudar a diferenciar as duas classes.


## 4. Tratamento de Dados Ausentes e Outliers

Agora que já vimos o retrato estatístico "antes" (Seção 3), temos a informação que faltava pra decidir como tratar tanto os dados ausentes (`state_of_health`, `cycle_count`, `internal_resistance`, `capacity_loss_percent`, `thermal_runaway_risk`, e as qualitativas) quanto os outliers das variáveis quantitativas. Depois do tratamento, vamos rodar a estatística descritiva mais uma vez ("depois") pra confirmar o efeito das escolhas e fechar o ciclo iterativo que começamos lá na abertura.


### Detecção de Outliers por IQR

A regra clássica marca como outlier qualquer ponto fora de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.


In [ ]:
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

resumo_outliers = {}
for col in VARS_QUANT:
    lim_inf, lim_sup = limites_iqr(df[col])
    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    resumo_outliers[col] = {
        "limite_inferior": round(lim_inf, 2),
        "limite_superior": round(lim_sup, 2),
        "n_outliers": n_outliers,
    }

pd.DataFrame(resumo_outliers).T


### Visualizando os Outliers


In [ ]:
fig, axes = plt.subplots(1, len(VARS_QUANT), figsize=(4 * len(VARS_QUANT), 5))

for i, col in enumerate(VARS_QUANT):
    sns.boxplot(data=df, y=col, color=COR_NEUTRA,
                flierprops=dict(markerfacecolor=COR_ALERTA, markeredgecolor=COR_ALERTA, marker="o"),
                ax=axes[i])
    axes[i].set_title(col)

fig.suptitle("Outliers pela regra do IQR (pontos em vermelho)", fontsize=14)
plt.tight_layout()
plt.show()


**Pra discutir:** o slide 26 da aula 9 classifica outliers em três tipos, cada um com um tratamento recomendado:

| Tipo | Cenário | O que fazer? |
|---|---|---|
| Estatístico | Legítimo | Manter / Transformar |
| De domínio | Viola regra de negócio ou condições físicas | Investigar! |
| Erros de dados | Falhas | Corrigir se possível / Remover como última opção |

Olhando os outliers encontrados (principalmente em `capacity_loss_percent` e `thermal_runaway_risk`): eles parecem plausíveis para baterias em fim de vida útil ou perto da falha (que é exatamente o fenômeno que o dataset tenta capturar), ou parecem violar algum limite físico conhecido (ex: percentuais negativos, scores fora de 0-100)? Isso muda qual das três categorias se aplica — e, consequentemente, se devemos manter, transformar ou remover. Vale lembrar: como o alvo é justamente `battery_failure`, é bem provável que os valores mais extremos sejam exatamente os casos de falha que queremos identificar — removê-los pode jogar fora a informação mais importante do dataset.


### Decidindo o Tratamento dos Dados Ausentes

Reunindo o que já discutimos: o mecanismo mais provável é MCAR (Seção 2), mas aqui o volume é bem diferente do Heart Disease. Só nas nossas 5 variáveis quantitativas de interesse, cerca de 19% das linhas têm pelo menos um valor ausente — remover todas essas linhas jogaria fora quase 1 em cada 5 veículos, um custo alto demais para um dataset cujo objetivo é justamente detectar uma classe minoritária (falhas, ~7% das linhas). Se olharmos as 70 colunas juntas, mais de 93% das linhas têm ao menos um valor ausente em algum lugar — remover linha por linha aqui destruiria o dataset.

Com mecanismo MCAR (ausência independente do valor) e volume alto, a decisão mais defensável é **imputar** em vez de remover: usamos a **mediana** para as quantitativas (robusta a assimetria, como vimos em `thermal_runaway_risk`) e a **moda** para as qualitativas. Isso preserva o tamanho da amostra e, como o mecanismo é aleatório, não deve introduzir viés sistemático.


In [ ]:
linhas_antes = df.shape[0]
df_antes = df.copy()

for col in VARS_QUANT:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)

for col in VARS_QUALI:
    moda = df[col].mode()[0]
    df[col] = df[col].fillna(moda)

print(f"Linhas antes: {linhas_antes} | Linhas depois: {df.shape[0]} (nenhuma linha removida — usamos imputação)")
print()
print("Valores ausentes restantes nas variáveis analisadas:")
print(df[VARS_QUANT + VARS_QUALI].isna().sum())


### Decidindo o Tratamento dos Outliers

Assim como no Heart Disease, aqui também **não vamos remover automaticamente**. Pelos valores encontrados, a classificação mais provável é **"Estatístico"** (outliers legítimos, não erros de coleta) — e, nesse dataset em particular, os valores mais extremos são justamente os candidatos mais fortes a serem os veículos com bateria em falha, que é a classe de interesse. O tratamento recomendado é **manter**, ou, se for necessário para alguma técnica estatística específica mais à frente (ex: métodos sensíveis à assimetria), **transformar**.

Vamos manter os valores originais por enquanto, mas ilustrar como ficaria uma transformação logarítmica em `thermal_runaway_risk` (nossa variável mais assimétrica), útil caso, mais adiante, a assimetria atrapalhe algum teste ou modelo.


In [ ]:
df["thermal_runaway_risk_log"] = np.log1p(df["thermal_runaway_risk"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=df, x="thermal_runaway_risk", kde=True, color=COR_NEUTRA, ax=axes[0])
axes[0].set_title(f"thermal_runaway_risk original (skew={df['thermal_runaway_risk'].skew():.2f})")

sns.histplot(data=df, x="thermal_runaway_risk_log", kde=True, color=COR_NEUTRA, ax=axes[1])
axes[1].set_title(f"log(1 + thermal_runaway_risk) (skew={df['thermal_runaway_risk_log'].skew():.2f})")

plt.tight_layout()
plt.show()


**Pra discutir:** a transformação logarítmica reduziu visivelmente a assimetria? Mantivemos `thermal_runaway_risk_log` só como demonstração, mas não vamos usá-la nas próximas seções. A ideia foi mostrar como opção se fosse o objetivo de, por exemplo, alimentar um teste paramétrico que exige normalidade.

**Nota para quando vocês replicarem no dataset de vocês:** nem todo outlier deve ser removido. Antes de decidir, classifiquem: é um valor estatisticamente raro mas plausível (manter/transformar), viola alguma regra física ou de negócio (investigar antes de decidir), ou é claramente um erro de digitação/coleta (corrigir ou remover)?


## 5. Estatística Descritiva pt 2

Fechando o ciclo iterativo: imputamos os ausentes e decidimos manter os outliers. Agora rodamos a mesma bateria de estatísticas da Seção 3 para comparar diretamente com o "antes".


In [ ]:
desc_depois = df[VARS_QUANT].describe().T
desc_depois["skewness"] = df[VARS_QUANT].skew()
desc_depois["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc_depois[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="Blues", subset=["skewness", "kurtosis"])
)


### Comparando "antes" x "depois"


In [ ]:
comparacao = pd.DataFrame({
    "skewness_antes": desc["skewness"],
    "skewness_depois": desc_depois["skewness"],
    "kurtosis_antes": desc["kurtosis"],
    "kurtosis_depois": desc_depois["kurtosis"],
})
comparacao["delta_skewness"] = (comparacao["skewness_depois"] - comparacao["skewness_antes"]).round(3)
comparacao["delta_kurtosis"] = (comparacao["kurtosis_depois"] - comparacao["kurtosis_antes"]).round(3)
comparacao.round(3)


**Pra discutir:** os valores mudaram muito? Diferente do Heart Disease (onde só ~2% das linhas foram removidas), aqui imputamos até ~5% de valores por coluna com a mediana — isso tende a "puxar" o centro da distribuição e pode reduzir um pouco o skewness/kurtosis de forma mais perceptível. Se a mudança for pequena mesmo assim, o que isso sugere sobre o quanto a imputação afetou a forma da distribuição original?


**O risco de thermal runaway ainda rejeita normalidade?**

Vamos rodar o mesmo Shapiro-Wilk da Seção 3, agora nos dados tratados.


In [ ]:
stat_sw_depois, p_valor_sw_depois = stats.shapiro(df["thermal_runaway_risk"].dropna())
print(f"Shapiro-Wilk (antes)  — thermal_runaway_risk: estatística={stat_sw:.4f}, p-valor={p_valor_sw:.4g}")
print(f"Shapiro-Wilk (depois) — thermal_runaway_risk: estatística={stat_sw_depois:.4f}, p-valor={p_valor_sw_depois:.4g}")


**Visualizando antes x depois, lado a lado**

Vamos sobrepor as distribuições de duas variáveis que mais chamaram atenção na Seção 4 (`thermal_runaway_risk` e `capacity_loss_percent`), comparando a versão original (`df_antes`) com a tratada (`df`).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ["thermal_runaway_risk", "capacity_loss_percent"]):
    sns.kdeplot(df_antes[col].dropna(), color=COR_CONTEXTO, fill=True, label="Antes", ax=ax)
    sns.kdeplot(df[col].dropna(), color=COR_DESTAQUE, fill=True, label="Depois", ax=ax)
    ax.set_title(col)
    ax.legend()

fig.suptitle("Distribuição Antes x Depois do Tratamento", fontsize=14)
plt.tight_layout()
plt.show()


**Pra discutir:** as curvas "antes" e "depois" praticamente se sobrepõem, ou dá pra notar um "pico" mais alto na mediana na curva "depois" (efeito típico de imputar vários valores no mesmo ponto)? Diferente do Heart Disease, onde a mudança era quase imperceptível, aqui o volume imputado (~5% por variável) deveria deixar essa diferença mais visível — essa checagem "antes x depois" é o que confirma (ou não) que o tratamento teve o efeito esperado, em vez de assumir isso às cegas.


## 6. Princípio Relacional


### Análise de Variáveis Qualitativas

Agora focaremos na avaliação de frequência e desbalanceamento das variáveis qualitativas.



#### Frequências absolutas e relativas


In [ ]:
for col in VARS_QUALI:
    freq_abs = df[col].value_counts()
    freq_rel = df[col].value_counts(normalize=True).round(3)
    print(f"\n--- {col} ---")
    print(pd.concat([freq_abs.rename("absoluta"), freq_rel.rename("relativa")], axis=1))


#### Visualizando as frequências


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharey=True)
axes = axes.ravel()

for i, col in enumerate(VARS_QUALI):
    df[col].value_counts().sort_index().plot(kind="bar", color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", rotation=30)

fig.suptitle("Frequências das Variáveis Qualitativas", fontsize=14)
plt.tight_layout()
plt.show()


**Pra discutir:** alguma variável qualitativa está bem desbalanceada (uma categoria dominando quase todas as observações)? Isso pode impactar como vamos interpretar cruzamentos com essa variável mais na frente na análise bivariada.


#### Desbalanceamento entre Classes

- **Razão de desbalanceamento** — # classe majoritária / # classe minoritária (quanto maior, mais desbalanceado).
- **Entropia de Shannon normalizada** — 1 = perfeitamente balanceado, 0 = uma classe domina tudo.

Vamos aplicar as duas em todas as qualitativas, incluindo o próprio alvo `status_bateria`.


In [ ]:
from scipy.stats import entropy

def metrica_desbalanceamento(serie):
    contagens = serie.value_counts()
    razao = contagens.max() / contagens.min()
    props = contagens / contagens.sum()
    n_classes = len(contagens)
    ent_normalizada = entropy(props, base=2) / np.log2(n_classes) if n_classes > 1 else np.nan
    return pd.Series({
        "n_classes": n_classes,
        "razao_desbalanceamento": round(razao, 2),
        "entropia_normalizada": round(ent_normalizada, 3),
    })

vars_para_checar = VARS_QUALI + ["status_bateria"]
pd.DataFrame({col: metrica_desbalanceamento(df[col]) for col in vars_para_checar}).T


**Pra discutir:** alguma variável está com entropia bem próxima de 0 (bem desbalanceada)? E o próprio alvo `status_bateria` — sabemos que só cerca de 7% dos veículos têm `battery_failure = 1` (uma razão de desbalanceamento de aproximadamente 13:1). Isso é bem diferente do Heart Disease, onde as duas classes eram relativamente equilibradas. Como esse desbalanceamento pode distorcer a análise bivariada (ex: médias e proporções dominadas pela classe majoritária) e um eventual modelo mais à frente?


### Análise Bivariada

Agora começamos a avaliar como as variáveis de fato se relacionam entre si e com o alvo `status_bateria`.


#### Quantitativa x Quantitativa - Correlação

Conhecemos três técnicas de correlação principais:
* **Pearson** (linear, sensível a outliers, exige normalidade),
* **Spearman** e **Kendall** (relação monotônica, não exigem normalidade).

Como já vimos que nenhuma das nossas variáveis passou no Shapiro-Wilk (e, com n≈19.000, isso é esperado — o teste fica hipersensível a desvios triviais nesse tamanho de amostra), Spearman é a escolha mais segura aqui, mas vamos calcular as três para comparar.


In [ ]:
corr_pearson = df[VARS_QUANT].corr(method="pearson")
corr_spearman = df[VARS_QUANT].corr(method="spearman")
corr_kendall = df[VARS_QUANT].corr(method="kendall")

fig, ax = plt.subplots(1,3,figsize=(14, 4),sharey=True)
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Matriz de Correlação — Pearson")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Matriz de Correlação — Spearman")
sns.heatmap(corr_kendall, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[2])
ax[2].set_title("Matriz de Correlação — Kendall")
plt.tight_layout()
plt.show()



Como os dados do triângulo superior e inferior são simétricos, poderíamos plotar a matriz de correlação da seguinte forma.

In [ ]:
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

plt.figure(figsize=(4, 3))
sns.heatmap(corr_spearman, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, vmin=-1, vmax=1,)
plt.grid(False)

plt.tight_layout()
plt.show()


**Pra discutir:** Pearson e Spearman divergem bastante em algum par de variáveis? Se sim, isso é um sinal de relação não-linear (monotônica, mas não em linha reta) em que Spearman capturaria e Pearson não. Também vale notar se `state_of_health`, `internal_resistance` e `capacity_loss_percent` aparecem fortemente correlacionadas entre si — o que seria esperado, já que as três descrevem, por ângulos diferentes, o mesmo processo de degradação da bateria.


#### Dispersão (scatter) entre duas variáveis quantitativas

Escolhemos `cycle_count` x `state_of_health` por um motivo de domínio, não só estatístico: existe uma relação eletroquímica bem documentada entre o número de ciclos de carga/descarga e a degradação da bateria, o que torna esse par um bom candidato pra visualizar uma tendência real.

**Nota para quando vocês replicarem no dataset de vocês:** nem sempre vai existir uma relação de domínio óbvia e, nesse caso, uma alternativa é escolher o par com o maior |correlação| na tabela acima, deixando a própria estatística guiar a escolha.


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="cycle_count", y="state_of_health", hue="status_bateria",
                palette={"Normal": COR_CONTEXTO, "Falha": COR_DESTAQUE}, alpha=0.6)
plt.title("Nº de Ciclos de Carga x Saúde da Bateria (state_of_health)")
plt.xlabel("Ciclos de Carga")
plt.ylabel("Saúde da Bateria (%)")
plt.tight_layout()
plt.show()



**Pra discutir:** existe uma tendência visível (conforme o número de ciclos aumenta, `state_of_health` tende a cair)? As duas classes ocupam regiões diferentes do gráfico — por exemplo, os pontos roxos ("Falha") concentrados nas saúdes mais baixas — ou estão misturadas?


#### Quantitativa x Qualitativa: Comparando Grupos

Na aula passada listamos **T-Test** (2 grupos) e **ANOVA** (3+ grupos) como testes paramétricos pra comparar uma variável quantitativa entre categorias, mas os dois assumem normalidade dentro de cada grupo.

Já vimos na Seção 3 (tabela generalizada do Shapiro-Wilk) que **nenhuma** das nossas quantitativas passou nesse teste. Isso significa que, tecnicamente, deveríamos usar os equivalentes **não-paramétricos**: **Mann-Whitney U** no lugar do T-Test, e **Kruskal-Wallis** no lugar da ANOVA. Vamos calcular os dois lado a lado pra comparar, mas o resultado não-paramétrico é o que deveríamos reportar como principal.


In [ ]:
resultados_comparacao = []
for col in VARS_QUANT:
    grupo_normal = df.loc[df["status_bateria"] == "Normal", col]
    grupo_falha = df.loc[df["status_bateria"] == "Falha", col]

    stat_t, p_t = stats.ttest_ind(grupo_normal, grupo_falha, equal_var=False)  # Welch (paramétrico)
    stat_u, p_u = stats.mannwhitneyu(grupo_normal, grupo_falha)                 # não-paramétrico

    resultados_comparacao.append({
        "variavel": col,
        "p_valor_ttest": round(p_t, 5),
        "p_valor_mannwhitney": round(p_u, 5),
        "concordam": (p_t < 0.05) == (p_u < 0.05),
    })

pd.DataFrame(resultados_comparacao).set_index("variavel")



**Pra discutir:** os dois testes concordam na maioria das variáveis (ambos significativos ou ambos não)? Quando divergem, é o **Mann-Whitney** que deveríamos priorizar, é o que respeita a premissa que de fato temos (dados não-normais), não a que gostaríamos de ter.

**Nota:** mantivemos o T-Test aqui só como contraste pedagógico. Reportar apenas o T-Test, sem essa ressalva, seria estatisticamente inconsistente com o que a própria Seção 3 já mostrou sobre a normalidade dessas variáveis.

*Exemplo com 3+ grupos — ANOVA x Kruskal-Wallis*

`terrain_type` (tipo de terreno onde o veículo roda) tem 5 categorias. Pelo mesmo motivo acima, calculamos a ANOVA (paramétrica) e o **Kruskal-Wallis** (não-paramétrico, equivalente pra 3+ grupos) lado a lado, testando se o risco térmico (`thermal_runaway_risk`) varia entre terrenos — uma hipótese plausível, já que terrenos desérticos ou montanhosos costumam impor mais estresse térmico à bateria.


In [ ]:
grupos_terreno = [grupo["thermal_runaway_risk"].values for _, grupo in df.groupby("terrain_type")]

stat_anova, p_anova = stats.f_oneway(*grupos_terreno)   # paramétrico
stat_kw, p_kw = stats.kruskal(*grupos_terreno)          # não-paramétrico

print(f"ANOVA          — thermal_runaway_risk entre categorias de terrain_type: estatística={stat_anova:.3f}, p-valor={p_anova:.5g}")
print(f"Kruskal-Wallis — thermal_runaway_risk entre categorias de terrain_type: estatística={stat_kw:.3f}, p-valor={p_kw:.5g}")

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="terrain_type", y="thermal_runaway_risk", color=COR_NEUTRA)
plt.title("Risco de Thermal Runaway por Tipo de Terreno")
plt.tight_layout()
plt.show()


**Pra discutir:** o Kruskal-Wallis concorda com a ANOVA sobre existir (ou não) diferença entre pelo menos uma categoria de `terrain_type`? Se o teste **não** encontrar diferença significativa, isso contraria a hipótese de domínio levantada acima (terrenos mais hostis gerando mais risco térmico) — o que pode significar que, neste dataset sintético, essa relação não foi modelada, ou que o efeito real existe mas é pequeno demais pra aparecer nesses dados.


#### Qualitativa x Qualitativa


In [ ]:
contingencia_quimica = pd.crosstab(df["battery_chemistry"], df["status_bateria"])
chi2, p_chi2, gl, esperado = stats.chi2_contingency(contingencia_quimica)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(contingencia_quimica, annot=True, fmt="d", cmap="magma", ax=ax)
ax.set_title(f"battery_chemistry x status_bateria (Qui² p-valor={p_chi2:.4g})")
plt.tight_layout()
plt.show()



**Pra discutir:** o p-valor do Qui-Quadrado indica que `battery_chemistry` e `status_bateria` são independentes, ou há associação estatisticamente significativa entre elas? Existe alguma química de bateria (NMC, NCA, LFP, LTO, LMO) com uma proporção de falha visivelmente maior que as outras no heatmap?


#### Avaliação de Fairness

Olhar o desbalanceamento da base inteira é só metade da história. Uma pergunta recorrente em discussões de **fairness** (justiça algorítmica) é: será que a distribuição do desfecho se comporta igual em diferentes subgrupos de atributos sensíveis? Aqui não temos atributos demográficos de pessoas, mas o equivalente mais próximo é: o veículo é de uso **fleet (frota) ou privado**, e a **faixa de idade do veículo**. Se não, isso pode refletir um padrão real (ex: veículos mais velhos falharem mais é fisicamente esperado) ou um viés na forma como os dados foram coletados (ex: veículos de frota são monitorados/manutenidos de forma diferente) — de qualquer forma, é algo que vale mapear **antes** de qualquer modelagem.


In [ ]:
df["faixa_idade_veiculo"] = pd.cut(df["vehicle_age_years"], bins=[0, 4, 7, 9, 100],
                                     labels=["<4 anos", "4-6 anos", "7-8 anos", "9+ anos"])

print("Desbalanceamento — fleet_or_private:")
print(metrica_desbalanceamento(df["fleet_or_private"]))
print("\nDesbalanceamento — faixa_idade_veiculo:")
print(metrica_desbalanceamento(df["faixa_idade_veiculo"].dropna()))


In [ ]:
prop_por_uso = pd.crosstab(df["fleet_or_private"], df["status_bateria"], normalize="index").round(3)
prop_por_idade = pd.crosstab(df["faixa_idade_veiculo"], df["status_bateria"], normalize="index").round(3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

prop_por_uso["Falha"].plot(kind="bar", color=COR_DESTAQUE, ax=axes[0])
axes[0].set_title("Proporção 'Falha' por Tipo de Uso")
axes[0].set_ylabel("Proporção")
axes[0].set_ylim(0, prop_por_uso["Falha"].max() * 1.5)
axes[0].tick_params(axis="x", rotation=0)

prop_por_idade["Falha"].plot(kind="bar", color=COR_DESTAQUE, ax=axes[1])
axes[1].set_title("Proporção 'Falha' por Faixa de Idade do Veículo")
axes[1].set_ylim(0, prop_por_idade["Falha"].max() * 1.5)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


**Pra discutir:** a proporção de falha varia bastante entre tipo de uso (fleet x privado) ou faixa de idade do veículo? Uma diferença por idade seria fisicamente esperada (baterias degradam com o tempo); já uma diferença por tipo de uso é mais difícil de justificar fisicamente e pode ser um sinal de viés de coleta (ex: frotas passam por manutenção/diagnóstico mais frequente, o que pode mudar a taxa de detecção de falha, não necessariamente a taxa real). A pergunta "a distribuição do desfecho é uniforme entre grupos?" é o ponto de partida de várias discussões de fairness em ciência de dados, e não tem uma resposta única: depende do contexto e da causa por trás da disparidade.


In [ ]:
for atributo in ["fleet_or_private", "faixa_idade_veiculo"]:
    contingencia = pd.crosstab(df[atributo], df["status_bateria"])
    chi2, p_valor, gl, esperado = stats.chi2_contingency(contingencia)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(contingencia, annot=True, fmt="d", cmap="magma", ax=ax)
    ax.set_title(f"{atributo} x status_bateria (Qui² p-valor={p_valor:.4g})")
    plt.tight_layout()
    plt.show()



**Pra discutir:** o Qui-Quadrado confirma associação estatisticamente significativa entre tipo de uso/faixa de idade e o desfecho? Lembrando que mesmo que a associação seja estatisticamente significativa, isso não diz *por que* ela existe — pode ser um fator de risco real (idade do veículo), pode ser um viés de quem/como os dados foram coletados (possivelmente tipo de uso), ou uma mistura das duas coisas. Um teste estatístico aponta *que* existe associação, não *a causa* dela — essa distinção é o núcleo de qualquer discussão séria de fairness.


#### Outliers bivariados


In [ ]:
sns.jointplot(data=df, x="internal_resistance", y="capacity_loss_percent", hue="status_bateria",
              palette={"Normal": COR_CONTEXTO, "Falha": COR_DESTAQUE}, alpha=0.5, height=6)
plt.suptitle("Resistência Interna x Perda de Capacidade: outliers conjuntos?", y=1.02)
plt.show()


**Pra discutir:** existe algum ponto que não parecia outlier nos histogramas univariados (Seção 3/4), mas que se destaca aqui como uma combinação incomum de `internal_resistance` e `capacity_loss_percent`? Esses pontos "duplamente extremos" tendem a ser majoritariamente da classe "Falha" (roxo), como seria de se esperar fisicamente?


#### Análise Multivariada


##### Matriz de Correlação


In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(df[VARS_QUANT].corr(method="spearman"), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de Correlação (Spearman) — Variáveis Quantitativas")
plt.tight_layout()
plt.show()



**Como interpretar:**


*   |r| < 0.3 ~ fraca,
*   0.3 < |r| < 0.7 ~ moderada,
*  |r| > 0.7 ~ forte.

Alguma correlação chama atenção por ser forte o suficiente pra sugerir redundância entre variáveis? (Dica: `state_of_health`, `internal_resistance` e `capacity_loss_percent` medem, por ângulos diferentes, o mesmo fenômeno de degradação — vale conferir se isso aparece como correlação muito forte entre elas.)


##### Correlograma


In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def circulo_correlacao(x, y, **kwargs):
    r = np.corrcoef(x, y)[0, 1]
    ax = plt.gca()
    ax.scatter(
        [0.5], [0.5],
        s=abs(r) * 4000,          # tamanho proporcional à força da correlação
        c=[r], cmap='coolwarm', vmin=-1, vmax=1,
        transform=ax.transAxes, edgecolor="white", linewidth=0.5,
    )
    ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes,
                ha="center", va="center", fontsize=10,
                color="white" if abs(r) > 0.5 else "black")
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)

g = sns.PairGrid(df[VARS_QUANT], diag_sharey=False)
g.map_lower(sns.scatterplot, color=COR_NEUTRA, alpha=0.5, s=20)
g.map_diag(sns.histplot, color=COR_NEUTRA, kde=True)
g.map_upper(circulo_correlacao)

# barra de cor única, compartilhada (cada subplot superior tem seu próprio scatter)
sm = ScalarMappable(norm=Normalize(-1, 1), cmap='coolwarm')
sm.set_array([])
g.fig.colorbar(sm, ax=g.axes, shrink=0.6, label="Correlação (r)")

g.fig.suptitle("Correlograma: Dispersão × Correlação", y=1.02, fontsize=15)


## Síntese Final

### Principais conclusões (preencher com 3 a 5 pontos)

1. *Preencher: o que mais chamou atenção na estrutura dos dados (linhas, colunas, ausentes)?*
2. *Preencher: qual foi a decisão de tratamento de ausentes/outliers e por quê?*
3. *Preencher: qual a relação mais forte encontrada na análise bivariada/multivariada?*
4. *Preencher: algum achado relevante de fairness (grupos com proporção de falha muito diferente)?*
5. *Preencher: alguma limitação do dataset ou da análise que vale registrar?*

### Declaração de Uso de IA

- **Ferramenta utilizada:** *(ex: Claude, ChatGPT, etc.)*
- **Para o que foi aplicada:** *(ex: corrigir erro de carregamento do dataset via kagglehub, adaptar o notebook de demonstração do Heart Disease para o dataset EV Battery Health, redigir respostas de discussão)*
- **O que foi aceito, modificado ou rejeitado, e por quê:** *(preencher com uma reflexão honesta sobre as sugestões da IA)*
